# Ганнова Анфиса - ТЗ
- https://hh.ru/resume/0204dee5ff0f1a4a6c0039ed1f374974704e65

* ###  SQL-запросы (4, 5, 6) - в конце блокнота
*  Google Sheets с результатами  [Реестр активных займов](https://docs.google.com/spreadsheets/d/1n8oCIAUg-hl5hnyyU6fZ9zVTvmwhW49y/edit?usp=sharing&ouid=107564873355874201227&rtpof=true&sd=true) страница-'Результаты'

# Задачи:
## 1. Составить реестр из всех клиентов отчёта 1

In [10]:
# Импорт библиотек
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import (PatternFill, Font, Alignment, Border, Side, numbers)
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.append('/Users/anfisso/Downloads/app')  # например: '/Users/username/Documents/my_scripts'
import f_  

FILE_PATH = "Fintech_ТЗ.xlsx"  # ← путь к исходному файлу

In [11]:
print("=" * 60)
print("1. Загрузка данных из Excel")
print("=" * 60)

xl = pd.ExcelFile(FILE_PATH)

df1 = pd.read_excel(xl, sheet_name="отчет 1")
df2 = pd.read_excel(xl, sheet_name="отчет 2")
df_cession = pd.read_excel(xl, sheet_name="цессия")

1. Загрузка данных из Excel


In [12]:
df1.head(3)

,№,Номер контракта,дата выдачи,Сумма выдачи,Задолженность по ОД,Задолженность по %%,"Кол-во дней просрочки, фактическое"
0,1,3547,2023-03-23,60000,47104.16,23926.86,525
1,2,68,2022-09-30,35000,11447.66,1300.06,552
2,3,389,2022-10-28,30000,4410.20,249.72,482


In [13]:
df2.head(3)

,NumContract,ContractID,ClientID,inner_lead_id,SubdivisionName,issueDateTimestamp,RepaymentPlanDate,LoanAmount,Status,DateStatus,SumLastPay
0,68,183,219,\N,Белгород,2022-09-30 15:18:31,2023-09-29,35000,COLLECTOR,2024-10-03 14:33:23,1950
1,91,252,291,\N,Электросталь,2022-10-04 17:40:57,2023-10-03,50000,EARLY_REPAID,2023-07-27 21:05:44,3078
2,265,774,954,\N,Вологда,2022-10-19 15:28:15,2023-06-28,25000,EARLY_REPAID,2023-05-16 16:30:05,5700


##  2. Недостающие поля взять из отчёта 2


In [14]:
print("\n" + "=" * 60)
print("2. Объединение отчётов (задачи 1 и 2)")
print("=" * 60)

# Переименуем колонки отчёта 2 для удобства
df2_renamed = df2.rename(columns={
    "NumContract": "Номер контракта",
    "ContractID": "ID договора",
    "SubdivisionName": "Регион выдачи",
    "issueDateTimestamp": "Дата выдачи (отч2)",
    "Status": "Статус",
    "DateStatus": "Дата последнего платежа",
    "SumLastPay": "Сумма последнего платежа",
})

# Берём нужные поля из отчёта 2
cols_from_df2 = [
    "Номер контракта",
    "ID договора",
    "Регион выдачи",
    "Статус",
    "Дата последнего платежа",
    "Сумма последнего платежа",
]
df2_slim = df2_renamed[cols_from_df2].copy()

# LEFT JOIN: всё из отчёта 1, дополняем данными из отчёта 2
registry = pd.merge(
    df1,
    df2_slim,
    on="Номер контракта",
    how="left"
)

# Переименуем «дата выдачи» из отчёта 1 → «Дата выдачи»
registry = registry.rename(columns={"дата выдачи": "Дата выдачи"})

# Финальный порядок колонок реестра (согласно заданию)
registry_columns = [
    "№",
    "Номер контракта",
    "ID договора",
    "Регион выдачи",
    "Дата выдачи",
    "Сумма выдачи",
    "Статус",
    "Задолженность по ОД",
    "Задолженность по %%",
    "Кол-во дней просрочки, фактическое",
    "Дата последнего платежа",
    "Сумма последнего платежа",
]
registry = registry[registry_columns]


2. Объединение отчётов (задачи 1 и 2)


In [9]:
f_.xlsx(registry, scroll=True, max_height=333)

№,Номер контракта,ID договора,Регион выдачи,Дата выдачи,Сумма выдачи,Статус,Задолженность по ОД,Задолженность по %%,"Кол-во дней просрочки, фактическое",Дата последнего платежа,Сумма последнего платежа
1,3547,10614,Электросталь,2023-03-23 00:00:00,60000,COLLECTOR,47104.16,23926.86,525,2023-11-21 23:00:00,1775
2,68,183,Белгород,2022-09-30 00:00:00,35000,COLLECTOR,11447.66,1300.06,552,2024-10-03 14:33:23,1950
3,389,1146,Вологда,2022-10-28 00:00:00,30000,DELINQUENT,4410.2,249.72,482,2023-10-07 00:00:00,1200
4,1393,4157,Электросталь,2022-12-20 00:00:00,40000,ACTIVE_SB,11088.24,1482.47,457,2023-12-12 23:00:00,3000
5,2050,6127,Санкт-Петербург,2023-01-26 00:00:00,50000,COLLECTOR,42219.24,30483.14,609,2024-12-17 15:34:37,2200
6,2154,6439,Курск,2023-01-31 00:00:00,20000,COLLECTOR,6606.81,864.92,583,2024-12-17 15:34:38,6652.05
7,2148,6421,Пенза,2023-01-31 00:00:00,30000,ACTIVE_SB,8796.44,1169.45,415,2024-01-23 23:00:00,9000
8,3211,9610,Белгород,2023-03-10 00:00:00,30000,COLLECTOR,25533.57,19657.32,573,2023-09-11 09:15:44,4250
9,3512,10509,Астрахань,2023-03-21 00:00:00,30000,COLLECTOR,26037.23,21091.66,569,2023-09-13 08:46:09,1100
10,6081,18155,Онлайн,2023-06-18 00:00:00,9500,COLLECTOR,9500.0,5157.97,578,2024-12-17 15:34:43,\N


##  3. Исключить клиентов из вкладки «цессия» 
##  и со статусами «банкрот», «погашен», «досрочно погашен»


In [15]:
print("\n" + "=" * 60)
print("3. Фильтрация реестра (задача 3)")
print("=" * 60)

# 3а — Цессия
cession_contract_ids = set(df_cession["ContractID"].dropna().astype(int))
print(f"ContractID в цессии: {sorted(cession_contract_ids)}")

mask_cession = registry["ID договора"].isin(cession_contract_ids)
excluded_cession = registry[mask_cession]
print(f"Исключено по цессии: {len(excluded_cession)} клиентов")
if len(excluded_cession):
    print(excluded_cession[["Номер контракта", "ID договора", "Регион выдачи", "Статус"]].to_string(index=False))

# 3б — Статусы (банкрот / погашен / досрочно погашен)
excluded_statuses = {"BANKRUPTCY", "REPAID", "EARLY_REPAID"}
mask_status = registry["Статус"].isin(excluded_statuses)
excluded_status_df = registry[mask_status & ~mask_cession]
print(f"\nИсключено по статусу ({excluded_statuses}): {len(excluded_status_df)} клиентов")
if len(excluded_status_df):
    print(excluded_status_df[["Номер контракта", "Статус"]].to_string(index=False))

# Итоговый реестр
registry_clean = registry[~mask_cession & ~mask_status].copy()
registry_clean = registry_clean.reset_index(drop=True)
registry_clean["№"] = registry_clean.index + 1          # перенумеровать


3. Фильтрация реестра (задача 3)
ContractID в цессии: [70784, 78335, 81233, 88259, 104036, 109897, 116791, 125922]
Исключено по цессии: 8 клиентов
 Номер контракта  ID договора Регион выдачи    Статус
           23629        70784        Онлайн COLLECTOR
           26146        78335        Онлайн COLLECTOR
           27112        81233        Онлайн COLLECTOR
           29454        88259        Онлайн COLLECTOR
           35084       104036        Онлайн COLLECTOR
           37654       109897        Онлайн COLLECTOR
           39952       116791        Онлайн COLLECTOR
           42997       125922        Онлайн COLLECTOR

Исключено по статусу ({'BANKRUPTCY', 'REPAID', 'EARLY_REPAID'}): 0 клиентов


In [16]:
f_.xlsx(registry_clean, scroll=True, max_height=333)

№,Номер контракта,ID договора,Регион выдачи,Дата выдачи,Сумма выдачи,Статус,Задолженность по ОД,Задолженность по %%,"Кол-во дней просрочки, фактическое",Дата последнего платежа,Сумма последнего платежа
1,3547,10614,Электросталь,2023-03-23 00:00:00,60000,COLLECTOR,47104.16,23926.86,525,2023-11-21 23:00:00,1775
2,68,183,Белгород,2022-09-30 00:00:00,35000,COLLECTOR,11447.66,1300.06,552,2024-10-03 14:33:23,1950
3,389,1146,Вологда,2022-10-28 00:00:00,30000,DELINQUENT,4410.2,249.72,482,2023-10-07 00:00:00,1200
4,1393,4157,Электросталь,2022-12-20 00:00:00,40000,ACTIVE_SB,11088.24,1482.47,457,2023-12-12 23:00:00,3000
5,2050,6127,Санкт-Петербург,2023-01-26 00:00:00,50000,COLLECTOR,42219.24,30483.14,609,2024-12-17 15:34:37,2200
6,2154,6439,Курск,2023-01-31 00:00:00,20000,COLLECTOR,6606.81,864.92,583,2024-12-17 15:34:38,6652.05
7,2148,6421,Пенза,2023-01-31 00:00:00,30000,ACTIVE_SB,8796.44,1169.45,415,2024-01-23 23:00:00,9000
8,3211,9610,Белгород,2023-03-10 00:00:00,30000,COLLECTOR,25533.57,19657.32,573,2023-09-11 09:15:44,4250
9,3512,10509,Астрахань,2023-03-21 00:00:00,30000,COLLECTOR,26037.23,21091.66,569,2023-09-13 08:46:09,1100
10,6081,18155,Онлайн,2023-06-18 00:00:00,9500,COLLECTOR,9500.0,5157.97,578,2024-12-17 15:34:43,\N


##  4. Построить сводную таблицу по регионам

In [17]:
print("\n" + "=" * 60)
print("4. Сводная таблица по регионам (задача 4)")
print("=" * 60)

pivot = (
    registry_clean
    .groupby("Регион выдачи", as_index=False)
    .agg(
        Кол_во_клиентов=("Номер контракта", "count"),
        Сумма_выданных_займов=("Сумма выдачи", "sum"),
    )
    .sort_values("Кол_во_клиентов", ascending=False)
    .reset_index(drop=True)
)
pivot.index = pivot.index + 1  # нумерация с 1

pivot.columns = ["Регион выдачи", "Количество клиентов", "Сумма выданных займов"]

# Итоговая строка
total_row = pd.DataFrame({
    "Регион выдачи":         ["ИТОГО"],
    "Количество клиентов":   [pivot["Количество клиентов"].sum()],
    "Сумма выданных займов": [pivot["Сумма выданных займов"].sum()],
})
pivot_with_total = pd.concat([pivot.reset_index(drop=True), total_row], ignore_index=True)


4. Сводная таблица по регионам (задача 4)


In [19]:
pivot_with_total

,Регион выдачи,Количество клиентов,Сумма выданных займов
0,Онлайн,5,44800
1,Белгород,2,65000
2,Электросталь,2,100000
3,Астрахань,1,30000
4,Вологда,1,30000
5,Курск,1,20000
6,Пенза,1,30000
7,Санкт-Петербург,1,50000
8,ИТОГО,14,369800


<!-- <div style='height:4rem'></div> -->
#  SQL

####  4. Клиенты с регионом выдачи «Онлайн», выданные в 2024 году

```
SELECT *
FROM   [отчет 2]
WHERE  LOWER(SubdivisionName) = 'онлайн'
  AND  YEAR(issueDateTimestamp) = 2024;
```

#### 5. Статус, количество клиентов в статусе, сумма выданных займов в статусе

```
SELECT
    Status                AS Статус,
    COUNT(*)              AS Количество_клиентов,
    SUM(LoanAmount)       AS Сумма_выданных_займов
FROM   [отчет 2]
GROUP BY Status
ORDER BY Количество_клиентов DESC;
```

#### 6. Все поля отчёта 1 и отчёта 2

```
SELECT
    r1.[№],
    r1.[Номер контракта],
    r2.[ContractID] AS [ID договора],
    r2.[ClientID],
    r2.[inner_lead_id],
    r2.[SubdivisionName] AS [Регион выдачи],
    r1.[дата выдачи] AS [Дата выдачи],
    r1.[Сумма выдачи],
    r2.[LoanAmount] AS [Сумма займа (отч2)],
    r2.[Status] AS [Статус],
    r2.[RepaymentPlanDate] AS [Плановая дата погашения],
    r2.[DateStatus] AS [Дата последнего платежа],
    r2.[SumLastPay] AS [Сумма последнего платежа],
    r1.[Задолженность по ОД],
    r1.[Задолженность по %%],
    r1.[Кол-во дней просрочки, фактическое]
FROM      [отчет 1] AS r1
LEFT JOIN [отчет 2] AS r2
       ON r1.[Номер контракта] = r2.[NumContract];
```

In [22]:
# Сохраняем данные
wb = load_workbook(FILE_PATH)
 
ws = wb.create_sheet("Результаты")
 
# Реестр
ws.append(["РЕЕСТР"])
ws.append(registry.columns.tolist())
for row in registry.itertuples(index=False):
    ws.append(list(row))
 
# Пустая строка между таблицами
ws.append([])
ws.append([])
 
# Сводная таблица
ws.append(["СВОДНАЯ ТАБЛИЦА"])
ws.append(pivot.columns.tolist())
for row in pivot.itertuples(index=False):
    ws.append(list(row))
 
wb.save(FILE_PATH)
print(f"✅ Лист 'Результаты' добавлен в {FILE_PATH}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
✅ Лист 'Результаты' добавлен в Fintech_ТЗ.xlsx
